# 🥈 Camada Silver — Limpeza, Tipagem e Deduplicação

## O que é a camada Silver?
A Silver transforma dados brutos em dados **confiáveis e consistentes**. Aqui aplicamos:
- **Tipagem correta** (converter strings para int, float, date)
- **Padronização** (uppercase, strip de espaços)
- **Tratamento de nulos** (decidir descartar, preencher ou sinalizar)
- **Deduplicação** (remover registros repetidos)
- **União de fontes** (combinar arquivos do mesmo domínio)

## Por que não fazemos isso na Bronze?
A Bronze é **imutável** — ela é a prova de que o dado chegou corretamente da fonte.
Se uma regra de limpeza estiver errada, podemos **reprocessar a partir da Bronze** a qualquer momento.
Separar as responsabilidades é o que torna o pipeline **auditável e recuperável**.

In [ ]:
# ── Configuração para Google Colab ────────────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# import os
# os.chdir('/content/drive/MyDrive/meu-projeto-medalhao')
# !pip install -q pandas pyarrow

import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

BRONZE = Path('../data/bronze')
SILVER = Path('../data/silver')
SILVER.mkdir(parents=True, exist_ok=True)

print('Ambiente pronto.')

## 1. Carregar dados da Bronze
Lemos os Parquets gerados no notebook anterior. Importante: ainda estão como `dtype=str`.

In [ ]:
df1 = pd.read_parquet(BRONZE / 'z0019_1.parquet')
df2 = pd.read_parquet(BRONZE / 'z0019_2.parquet')

print(f'Bronze 1: {df1.shape}')
print(f'Bronze 2: {df2.shape}')

print('\n=== HEAD — Bronze 1 (entrada Silver) ===')
display(df1.head())

## 2. União das fontes (UNION ALL)
Os dois arquivos representam o **mesmo domínio** (estoque de materiais), portanto
fazemos `pd.concat` — equivalente ao `UNION ALL` do SQL.

> **Por que UNION ALL e não JOIN?** São cargas parciais do mesmo dataset.
> Um JOIN seria usado se fossem dimensões diferentes (ex: material + fornecedor).

In [ ]:
# Colunas de auditoria Bronze — não fazem parte do modelo de negócio Silver
AUDIT_COLS = ['_source_file', '_ingested_at', '_row_hash']

df_union = pd.concat([df1, df2], ignore_index=True)

print(f'Total após união: {len(df_union)} linhas')
print(f'De: {df_union["_source_file"].value_counts().to_dict()}')

display(df_union.head(6))

## 3. Relatório de Qualidade (antes da limpeza)
Documentar a qualidade **antes** é importante: permite comparar com o estado **após**
e justificar cada decisão de limpeza.

In [ ]:
business_cols = [c for c in df_union.columns if not c.startswith('_')]
df_biz = df_union[business_cols]

print('=== Qualidade ANTES da limpeza ===')
quality_report = pd.DataFrame({
    'tipo_raw'    : df_biz.dtypes,
    'nulos'       : df_biz.isnull().sum(),
    'pct_null'    : (df_biz.isnull().sum() / len(df_biz) * 100).round(1),
    'unicos'      : df_biz.nunique(),
    'exemplo'     : df_biz.iloc[0],
})
display(quality_report)

# Verificar duplicatas pela chave de negócio esperada
dupes = df_biz.duplicated(subset=['NATB', 'WERKS']).sum()
print(f'\nDuplicatas por (NATB + WERKS): {dupes}')

## 4. Limpeza e Padronização

### 4a. Padronização de strings

In [ ]:
df_silver = df_union.copy()

# MAKTX: remover espaços extras e padronizar para UPPER
# Por quê? Evita duplicatas silenciosas: 'Parafuso' ≠ 'PARAFUSO' ≠ ' PARAFUSO'
df_silver['MAKTX'] = df_silver['MAKTX'].str.strip().str.upper()

# WERKS e MAINS: também padronizar (são códigos, devem ser uppercase)
df_silver['WERKS'] = df_silver['WERKS'].str.strip().str.upper()
df_silver['MAINS'] = df_silver['MAINS'].str.strip()

print('Strings padronizadas (strip + upper).')
display(df_silver[business_cols].head())

### 4b. Conversão de Tipos
Na Bronze, tudo era `str`. Agora atribuímos os tipos corretos de negócio.

In [ ]:
# NATB → int (código de material numérico)
# errors='coerce' converte valores inválidos para NaN em vez de quebrar
df_silver['NATB']  = pd.to_numeric(df_silver['NATB'],  errors='coerce').astype('Int64')

# MAINS → int (tipo de estoque numérico)
df_silver['MAINS'] = pd.to_numeric(df_silver['MAINS'], errors='coerce').astype('Int64')

# LABST → float (quantidade em estoque pode ter decimais)
df_silver['LABST'] = pd.to_numeric(df_silver['LABST'], errors='coerce').astype(float)

print('Tipos após conversão:')
print(df_silver[business_cols].dtypes)

# Verificar se a conversão gerou novos nulos (indicaria dados inválidos na fonte)
novos_nulos = df_silver[business_cols].isnull().sum()
print(f'\nNulos após conversão (novos nulos = dado inválido na fonte):')
print(novos_nulos)

### 4c. Deduplicação
Os dois arquivos contêm o material `1003/PREGO`. Qual manter?
**Regra de negócio:** mantemos o registro mais recente (`keep='last'`) pois
assumimos que a segunda carga é uma atualização de estoque.

In [ ]:
print('Antes da deduplicação:')
dupes_df = df_silver[df_silver.duplicated(subset=['NATB', 'WERKS'], keep=False)]
display(dupes_df[business_cols + ['_source_file']])

# Deduplicação: chave = (material + planta)
# keep='last' = preserva o registro mais recente da segunda carga
df_silver = df_silver.drop_duplicates(subset=['NATB', 'WERKS'], keep='last')

print(f'\nApós deduplicação: {len(df_silver)} linhas')

### 4d. Adicionar metadados Silver

In [ ]:
df_silver['_processed_at'] = datetime.utcnow().isoformat()
df_silver['_schema_version'] = '1.0'

# Remover colunas de auditoria Bronze — já não são necessárias na Silver
# Mantemos _source_file para rastreabilidade, removemos as demais
df_silver = df_silver.drop(columns=['_ingested_at', '_row_hash'])

print('Schema final Silver:')
print(df_silver.dtypes)

## 5. Persistir na Silver

In [ ]:
silver_path = SILVER / 'estoque_materiais.parquet'
df_silver.to_parquet(silver_path, index=False)
print(f'[Silver] ✓ Gravado → {silver_path}')
print(f'          {len(df_silver)} linhas | {silver_path.stat().st_size} bytes')

## 6. Head() — Visualização do Estado Silver
> **O que mudou?** Strings padronizadas, tipos corretos, 1 duplicata removida (PREGO — mantemos o estoque atualizado = 60 em vez de 50), metadados Silver adicionados.

In [ ]:
print('=== HEAD — Camada Silver ===')
display(df_silver.reset_index(drop=True))

print('\n=== Relatório de Qualidade APÓS limpeza ===')
quality_after = pd.DataFrame({
    'tipo_silver' : df_silver[[c for c in business_cols if c in df_silver.columns]].dtypes,
    'nulos'       : df_silver[[c for c in business_cols if c in df_silver.columns]].isnull().sum(),
})
display(quality_after)

## Resumo da Camada Silver

| Transformação | Antes | Depois |
|---|---|---|
| Linhas totais | 6 | 5 |
| Duplicatas (NATB+WERKS) | 1 | 0 |
| Tipos de dados | tudo `str` | tipos corretos |
| Strings padronizadas | não | UPPER + strip |
| Schema versionado | não | v1.0 |

**Próximo passo:** Notebook `03_transformacao_gold.ipynb` — agregações e modelo Star Schema.